# Notebook 05 — Crypto Derivatives: Deribit BTC/ETH Calibration

Fetches a live BTC or ETH option snapshot from Deribit REST API,
builds an IV surface, and calibrates the Rough Heston model.

**Runtime estimate:** 1–3 min (live network required)

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src") if os.path.basename(os.getcwd()) == "notebooks"
                else os.path.join(os.getcwd(), "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "font.family": "serif",
})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


from market.deribit_data import fetch_option_snapshot, build_iv_surface, calibrate_crypto
from fno_model import MirrorPaddedFNO2d
from calibrate import _load_normalizers


## 1. Fetch Live Option Snapshot from Deribit

In [ ]:
CURRENCY = "BTC"   # or "ETH"

# Jupyter already runs an asyncio event loop — use 'await' directly
df = await fetch_option_snapshot(CURRENCY)
print(f"Fetched {len(df)} live {CURRENCY} options from Deribit")
print(f"Columns: {df.columns.tolist()}")
print(df[["instrument_name","expiry","strike","mark_iv","log_moneyness"]].head(10).to_string())


## 2. Build the IV Surface

In [ ]:
from market.spx_data import T_GRID, K_GRID
iv_surface = build_iv_surface(df, currency=CURRENCY)   # shape (8, 11)
T_nodes, K_nodes = T_GRID, K_GRID
print(f"Maturities (yr): {np.round(T_nodes, 3)}")
print(f"Log-moneyness:   {np.round(K_nodes, 2)}")
print(f"ATM vol range:   {iv_surface[:, 5].min()*100:.1f}% — "
      f"{iv_surface[:, 5].max()*100:.1f}%")


## 3. Plot the Live IV Surface

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(iv_surface * 100, aspect="auto", cmap="RdYlGn_r",
               extent=[K_nodes[0], K_nodes[-1], T_nodes[-1], T_nodes[0]])
ax.set_xlabel("Log-moneyness"); ax.set_ylabel("Maturity (yr)")
ax.set_title(f"Live {CURRENCY} IV Surface (Deribit)")
plt.colorbar(im, label="Implied Vol (%)"); plt.tight_layout(); plt.show()


## 4. Calibrate Rough Heston to Crypto

Uses the dedicated `calibrate_crypto()` function which handles the
parameter range adjustments needed for BTC/ETH (higher vol-of-vol, wider grid).

In [ ]:
# calibrate_crypto handles model loading and normalizer setup internally
result = calibrate_crypto(currency=CURRENCY)
print(f"\nCalibration RMSE: {result['rmse_bps']:.1f} bps")
print(f"Converged:        {result.get('converged', 'N/A')}")
print(f"Params clipped:   {result.get('params_clipped', False)}")
for name in ['v0', 'sigma', 'rho', 'zeta', 'lambda', 'H']:
    if name in result:
        print(f"  {name:6s} = {result[name]:.4f}")


## 5. Market vs Model Overlay

In [ ]:
pred = result.get("iv_fitted")   # (8,11) model-predicted surface from calibrate_newton_h
if pred is not None and pred.shape == iv_surface.shape:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, data, title in zip(axes,
            [iv_surface*100, pred*100],
            [f"{CURRENCY} Market IV (%)", "Rough Heston Model IV (%)"]):
        im = ax.imshow(data, aspect="auto", cmap="RdYlGn_r",
                       extent=[K_nodes[0], K_nodes[-1], T_nodes[-1], T_nodes[0]])
        ax.set_title(title); ax.set_xlabel("Log-moneyness")
        ax.set_ylabel("Maturity (yr)")
        plt.colorbar(im, ax=ax, fraction=0.04)
    plt.tight_layout(); plt.show()
    residuals = (pred - iv_surface) * 1e4
    print(f"Max |residual|: {np.abs(residuals).max():.1f} bps")
else:
    print("Model surface not available — check calibrate_crypto return value")
